# Lab 6: Recurrent Neural Networks (RNNs) in PyTorch

Objectives: By the end of this lab, students should be able to:
(1) Understand sequence data and RNN intuition, (2) Implement an RNN using PyTorch, 
(3) Train an RNN for sequence classification, and (4) Evaluate and improve model performance

## Task Overview:
    You will build a model that classifies sentences as positive or negative sentiment using an RNN.

### Part 1: Dataset Preparation
We will use a toy dataset (for simplicity).

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

# Data
sentences = [
"i love this movie",
"this film is great",
"amazing experience",
"i hate this movie",
"this film is terrible",
"bad experience"
]

labels = torch.tensor([1, 1, 1, 0, 0, 0])

# Tokenization
tokenized = [s.split() for s in sentences]

# Build vocab
vocab = {"<pad>": 0}
for sentence in tokenized:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

# Convert to indices
indexed = [
torch.tensor([vocab[word] for word in sentence])
for sentence in tokenized
]

# Pad sequences
padded = pad_sequence(indexed, batch_first=True, padding_value=0)

print("Vocab:", vocab)
print("Padded shape:", padded.shape)

In [ ]:
print(padded)

### Part 2: Build the RNN Model

Implement an RNN classifier using:

Embedding layer

RNN layer

Fully connected layer


In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.hh = nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
        self.xh = nn.Linear(in_features=embed_dim, out_features=hidden_dim)
        self.a = nn.ReLU()
        self.o = nn.Linear(in_features=hidden_dim, out_features=output_dim)

    def forward(self, x: torch.Tensor):
        x = self.embedding(x) # n, t, embed_dim
        latent = torch.zeros(x.shape[0], self.hidden_dim, device=x.device)
        for i in range(x.shape[1]):
            h = self.hh(latent)
            k = self.xh(x[:, i, :])
            latent = self.a(h + k)

        return self.o(latent)
        

# Hyperparameters
vocab_size = len(vocab)
embed_dim = 10
hidden_dim = 16
output_dim = 2

model = RNNClassifier(vocab_size, embed_dim, hidden_dim, output_dim)


In [ ]:
from flax import nnx

nnx.Embed()

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Parameter(torch.normal(0, 1, size=(vocab_size, embed_dim)))
        self.hh = nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
        self.xh = nn.Linear(in_features=embed_dim, out_features=hidden_dim)
        self.a = nn.ReLU()
        self.o = nn.Linear(in_features=hidden_dim, out_features=output_dim)

    def forward(self, x: torch.Tensor):
        x = self.embedding[x] # n, t, embed_dim
        latent = torch.zeros(x.shape[0], self.hidden_dim, device=x.device)
        for i in range(x.shape[1]):
            h = self.hh(latent)
            k = self.xh(x[:, i, :])
            latent = self.a(h + k)

        return self.o(latent)

In [ ]:
# Hyperparameters
vocab_size = len(vocab)
embed_dim = 10
hidden_dim = 16
output_dim = 2

model = RNNClassifier(vocab_size, embed_dim, hidden_dim, output_dim)

### Part 3: Training

Train the model using:

Loss: CrossEntropyLoss

Optimizer: Adam

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(padded)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


### Part 4: Evaluation

Evaluate accuracy on the training set.

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    outputs = model(padded)
    # Complete the following two lines:
    predictions = F.softmax(outputs, dim=1).argmax(1)
    accuracy = (predictions == labels).sum() / len(labels)

print("Predictions:", predictions)
print("Accuracy:", accuracy.item())

### Part 5: 

Replace the above RNN with LSTM.